In [1]:
import torch
import torch.nn as nn

In [ ]:
batch_size = 64
sequence_len = 64
embedding_size = 32
head_size = 256
steps = 5000

In [3]:
# input & output initialization
with open("inputs-shakespeare.txt", "r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(chars)}
itoc = dict(enumerate(chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, sequence_len, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(0, sequence_len, 1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, sequence_len, sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(0, sequence_len, sequence_len)

In [ ]:
# model declaration
class RecurrentNeuralNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_size, head_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.input_linear = nn.Linear(in_features=embedding_size, out_features=head_size)
        self.hidden_linear = nn.Linear(in_features=head_size, out_features=head_size)
        self.output_linear = nn.Linear(in_features=head_size, out_features=vocab_size)        

    def forward(self, input, h=None):
        batch_size = input.size(0)
        if h is None:
            h = torch.zeros((batch_size, head_size))
        logits = []
        for i in range(input.size(1)):
            ch = input[:, i]
            new_h = torch.tanh(self.input_linear(self.embedding(ch)) + self.hidden_linear(h))
            logits.append(self.output_linear(new_h))
            h = new_h
        return torch.stack(logits, dim=1), h

In [ ]:
# model initialization
model = RecurrentNeuralNetwork(vocab_size, embedding_size, head_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()
    
    batch = torch.randint(0, len(x_train), (batch_size,))
    train_logits, _ = model(x_train[batch])
    train_loss = nn.functional.cross_entropy(train_logits.view(-1, vocab_size), y_train[batch].view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits, _ = model(x_val)
            val_loss = nn.functional.cross_entropy(val_logits.view(-1, vocab_size), y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.153553485870361, val loss = 4.155017852783203
Step 1000: train loss = 1.5627119541168213, val loss = 1.7138535976409912
Step 2000: train loss = 1.4475739002227783, val loss = 1.6624107360839844
Step 3000: train loss = 1.4387398958206177, val loss = 1.6501789093017578
Step 4000: train loss = 1.4371222257614136, val loss = 1.6427611112594604
Step 4999: train loss = 1.4778656959533691, val loss = 1.6309940814971924


In [7]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000
    h = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)            

    while len(output) < max_output_length:
        logits, h = model(i, h)

        P = torch.softmax(logits[0, -1, :], dim=0)
        j = torch.multinomial(P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long)

In [8]:
print(''.join(output))

ROMEO:

CLAUDIO:
Your breast, call bloody and as then?

Provost:
And curcaSity, that careing!
Lord's
More heart you, cause of tenderstrets, if Gentleman;
Nay, and not, my ground before nevice wanted,
If non, the good
Without further things of this present.

KING RICHARD III:
There we say,
And served by say you, tom upon of request; I'll be friend of him: i' the empty laby like afflcting al mine heart, is courteniague to my pale: pity are royal I off, his beght! nake here command.

RIVERS:
Ia set you, like you;
Hont by our husband
With the great the like up in
by Juliet! I prouddition; but in Lady Pariin: I would braven die;
I am you;
Hath's that wounded, thou beroun my wealth.

ABHORSN:
Your sweet learn that full prote knowledge:
Yot call it wondly makenge,
Iw does.
Lord, yet
itlong it out on into my import. Doppening to be enhur.

Lanth.

Procet who in persed frothers of my softles, Marcius,
As thou despise the hosion;
Our daughter is, he should up your poor pavellitious good some cou